In [1]:
# Galaxies whose final BBarolo fit has 4 rings or fewer: how many, and why.
# "Rings" = the rings BBarolo actually fitted (rows of bbarolo/rings_final2.txt), not the NRADII it was asked for.
import json
import numpy as np
import pandas as pd
from pathlib import Path
from astropy.io import fits

MARTINI = Path("/scratch/s4636708/aida/derived/martini")
SNAP_Z = {17: 5, 21: 4, 25: 3, 33: 2, 50: 1, 67: 0.5}
PX_ARCSEC = 5 # pixel size
RING_ARCSEC = 30 # ring width (= beam)
BEAM_PX = 6 # 30 arcsec beam / 5 arcsec pixels
KPC_PER_ARCSEC = 5000 / 206265 # the galaxies are placed at 5 Mpc: 1 arcsec = 0.0242 kpc, one ring = 0.73 kpc
FEW = 4 # "few rings" = 4 or fewer, i.e. the walk stopped before 150 arcsec (3.6 kpc)
TAIL = 0.10 # new rule: a gap of a beam or more is stepped over if more than 10% of the side's flux lies beyond it

In [ ]:
def gal_dir(model, z, sub):
    return MARTINI / f"z{z:g}" / model / f"gal_{sub:06d}"

def major_axis(model, z, sub):
    """mom1 and mom0 along the major axis: (left side, right side), each from the centre outward.
    Same maps and row as the pipeline's walk (barolo.major_axis_extent): the DATA moment maps of the
    3-ring fit, row through the cube centre (PA = 90 deg, so the major axis is horizontal)."""
    maps = gal_dir(model, z, sub) / "bbarolo_3rings" / "maps"
    mom1 = np.squeeze(fits.getdata(maps / "MOCK_1mom.fits"))
    mom0 = np.squeeze(fits.getdata(maps / "MOCK_0mom.fits"))
    ny, nx = mom1.shape
    x0, y0 = int((nx - 1) / 2), int((ny - 1) / 2) # centre pixel, as in the walk
    return (mom1[y0, x0::-1], mom0[y0, x0::-1]), (mom1[y0, x0:], mom0[y0, x0:])

def walk_now(mom1):
    """Current rule: skip the blank centre (hole), then stop at the first blank pixel.
    A pixel has emission if its mom1 is not NaN, i.e. it is inside BBarolo's mask.
    Returns (first emission, last emission) in pixels from the centre, and why the walk stopped:
    'edge' = cube border, 'end' = no emission further out,
    'gap<beam' / 'gap>=beam' = there is emission after the gap, which is narrower / not narrower than a beam.
    None if this side has no emission."""
    has = np.isfinite(mom1)
    if not has.any():
        return None
    start = int(np.argmax(has)) # first pixel with emission
    blank = np.flatnonzero(~has[start:])
    if len(blank) == 0:
        return start, len(has) - 1, "edge"
    g0 = start + int(blank[0]) # first blank pixel: start of a gap
    after = np.flatnonzero(has[g0:])
    if len(after) == 0:
        return start, g0 - 1, "end"
    return start, g0 - 1, "gap<beam" if after[0] < BEAM_PX else "gap>=beam"

def walk_new(mom1, mom0, tail=TAIL):
    """New rule (task 6): skip the blank centre; step over gaps narrower than a beam; step over a gap of a beam
    or more if more than `tail` of this side's flux (mom0 along the row) lies beyond it, otherwise stop there.
    Returns (first emission, last emission, why it stopped: 'edge', 'end' or 'tail'), or None.
    tail=np.inf: step over the gaps narrower than a beam only."""
    has = np.isfinite(mom1)
    flux = np.nan_to_num(mom0)
    if not has.any():
        return None
    start = i = int(np.argmax(has))
    while True:
        blank = np.flatnonzero(~has[i:])
        if len(blank) == 0:
            return start, len(has) - 1, "edge"
        g0 = i + int(blank[0])
        after = np.flatnonzero(has[g0:])
        if len(after) == 0:
            return start, g0 - 1, "end"
        width = int(after[0]) # gap width in pixels
        beyond = flux[g0:][has[g0:]].sum() / flux[has].sum() # share of this side's flux beyond the gap
        if width < BEAM_PX or beyond > tail:
            i = g0 + width # step over the gap
            continue
        return start, g0 - 1, "tail"

def fitted_radii(model, z, sub):
    """Radii (arcsec) of the rings BBarolo fitted in the final fit, or None if there is no rings file."""
    f = gal_dir(model, z, sub) / "bbarolo" / "rings_final2.txt"
    if not f.exists():
        return None
    rows = [l.split() for l in f.read_text().splitlines() if l.strip() and not l.startswith("#")]
    return np.array([float(r[1]) for r in rows])

In [3]:
# every galaxy in the manifest (~5 min; reading the files from /scratch is the slow part, so 8 galaxies are read at a time)
from concurrent.futures import ThreadPoolExecutor

def measure(line):
    """The row of one galaxy (manifest line: model, snap, subID), or None if it has no moment maps or info.json."""
    model, snap, sub = line.split()[:3]
    z, sub = SNAP_Z[int(snap)], int(sub)
    try:
        sides = major_axis(model, z, sub)
        info = json.loads((gal_dir(model, z, sub) / "info.json").read_text())
    except OSError:
        print("no moment maps or info.json:", model, z, sub)
        return None
    now = [walk_now(m1) for m1, m0 in sides] # current rule, (left, right)
    new = [walk_new(m1, m0) for m1, m0 in sides] # new rule
    bridge = [walk_new(m1, m0, tail=np.inf) for m1, m0 in sides] # new rule, but never stepping over a gap of a beam or more
    fitted = fitted_radii(model, z, sub)
    asked = RING_ARCSEC * (np.arange(info["nradii"]) + 0.5) # rings asked of BBarolo now: 15, 45, 75, ... arcsec
    emission = [w for w in new if w is not None] # sides with emission
    r_in = max(w[0] for w in emission) * PX_ARCSEC if emission else 0.0 # new rule: larger first-emission radius
    r_out = max(w[1] for w in emission) * PX_ARCSEC if emission else 0.0 # larger last-emission radius
    has_rings = fitted is not None and len(fitted) > 0
    return dict(model=model, z=z, subID=sub, bbarolo_rc=info.get("bbarolo_rc"), nradii=info["nradii"],
                n_fit=np.nan if fitted is None else len(fitted),
                # asked rings missing inside the first fitted ring / beyond the last fitted ring
                dropped_inner=int((asked < fitted.min() - 1).sum()) if has_rings else np.nan,
                dropped_outer=int((asked > fitted.max() + 1).sum()) if has_rings else np.nan,
                why_now=[None if w is None else w[2] for w in now],
                extent_now=max(0 if w is None else w[1] for w in now) * PX_ARCSEC,
                extent_bridge=max(0 if w is None else w[1] for w in bridge) * PX_ARCSEC,
                why_new=[None if w is None else w[2] for w in new],
                starts=[None if w is None else w[0] * PX_ARCSEC for w in new], # first emission, arcsec (left, right)
                stops=[None if w is None else w[1] * PX_ARCSEC for w in new], # last emission, new rule
                r_in_new=r_in, r_out_new=r_out,
                n_new=max(1, int((r_out - r_in) // RING_ARCSEC)))

with ThreadPoolExecutor(8) as pool:
    rows = [r for r in pool.map(measure, open(MARTINI / "manifest.txt")) if r is not None]
df = pd.DataFrame(rows)
# check: my copy of the current walk gives the pipeline's NRADII (info.json) for every galaxy
print("NRADII reproduced for", (df["nradii"] == np.maximum(1, df["extent_now"] // RING_ARCSEC)).sum(), "of", len(df))
df

no moment maps or info.json: SIDM1 3 14303
NRADII reproduced for 4081 of 4081


,model,z,subID,bbarolo_rc,nradii,n_fit,dropped_inner,dropped_outer,why_now,extent_now,extent_bridge,why_new,starts,stops,r_in_new,r_out_new,n_new
0,CDM,5.0,1202,0.0,6,6.0,0.0,0.0,"[end, end]",195,195,"[end, end]","[0, 0]","[175, 195]",0.0,195.0,6
1,CDM,4.0,668,0.0,14,14.0,0.0,0.0,"[end, gap<beam]",430,430,"[end, end]","[0, 0]","[430, 250]",0.0,430.0,14
2,CDM,4.0,1056,0.0,5,5.0,0.0,0.0,"[end, end]",160,160,"[end, end]","[0, 0]","[160, 140]",0.0,160.0,5
3,CDM,4.0,2028,0.0,5,5.0,0.0,0.0,"[end, gap<beam]",170,215,"[end, tail]","[0, 0]","[155, 215]",0.0,215.0,7
4,CDM,4.0,1724,0.0,9,9.0,0.0,0.0,"[end, end]",295,295,"[end, end]","[0, 0]","[255, 295]",0.0,295.0,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4076,vSIDM,0.5,75970,0.0,27,27.0,0.0,0.0,"[end, end]",815,815,"[end, end]","[0, 0]","[815, 620]",0.0,815.0,27
4077,vSIDM,0.5,77196,0.0,15,15.0,0.0,0.0,"[gap<beam, end]",475,535,"[end, end]","[0, 0]","[535, 430]",0.0,535.0,17
4078,vSIDM,0.5,71107,0.0,20,20.0,0.0,0.0,"[gap>=beam, gap>=beam]",605,605,"[tail, tail]","[0, 0]","[435, 605]",0.0,605.0,20
4079,vSIDM,0.5,82374,0.0,9,9.0,0.0,0.0,"[gap>=beam, end]",290,290,"[tail, end]","[0, 0]","[290, 280]",0.0,290.0,9


In [4]:
# how many galaxies have 4 fitted rings or fewer
#   N: galaxies; few: 4 fitted rings or fewer; pct: few / N in %
#   no_rings_file: the final fit wrote no rings file (not counted in few)
#   median_extent_kpc: median of the larger major-axis extent with the new walk, in kpc (150 arcsec = 3.6 kpc)
def count(g):
    few = g["n_fit"] <= FEW
    return pd.Series(dict(N=len(g), few=few.sum(), pct=round(100 * few.mean(), 1),
                          no_rings_file=g["n_fit"].isna().sum(),
                          median_extent_kpc=round(g["r_out_new"].median() * KPC_PER_ARCSEC, 1)), dtype=object)

by_z = df.groupby("z").apply(count)
by_z.loc["all"] = count(df)
by_z

,N,few,pct,no_rings_file,median_extent_kpc
z,,,,,
0.5,1424,73,5.1,1,16.2
1.0,1339,51,3.8,0,11.5
2.0,854,41,4.8,0,7.6
3.0,376,31,8.2,0,6.4
4.0,80,5,6.2,0,5.6
5.0,8,2,25.0,0,4.3
all,4081,203,5.0,1,10.5


In [5]:
# the same per redshift and model
df.groupby(["z", "model"]).apply(count)

N  few   pct  no_rings_file  median_extent_kpc
z   model                                                  
0.5 CDM    460   27   5.9              0               16.4
    SIDM1  503   25   5.0              0               15.3
    vSIDM  461   21   4.6              1               16.5
1.0 CDM    425   20   4.7              0               11.8
    SIDM1  477   13   2.7              0               11.0
    vSIDM  437   18   4.1              0               11.8
2.0 CDM    259   14   5.4              0                7.5
    SIDM1  312   16   5.1              0                7.8
    vSIDM  283   11   3.9              0                7.8
3.0 CDM    122    5   4.1              0                6.9
    SIDM1  138    9   6.5              0                6.5
    vSIDM  116   17  14.7              0                6.2
4.0 CDM     23    0   0.0              0                5.6
    SIDM1   33    4  12.1              0                5.7
    vSIDM   24    1   4.2              0                5.6
5.0 CDM      1    0   0.0              0                4.7
    SIDM1    4    1  25.0              0                3.9
    vSIDM    3    1  33.3              0                4.4

In [6]:
# why: each galaxy with 4 fitted rings or fewer gets the first reason that applies
#   no_gas:      no emission on the major axis at all
#   dropped:     the walk reached 150 arcsec (it asked for 5 rings or more) but BBarolo fitted 4 or fewer
#   gap_lt_beam: the walk stopped before 150 arcsec at a gap narrower than a beam; stepping over such gaps takes it past 150 arcsec
#   gap_ge_beam: the same, but it also has to step over a gap of a beam or more with more than 10% of the flux beyond
#   tail:        even the new walk stops before 150 arcsec, at a gap of a beam or more with 10% or less of the flux beyond
#   ends:        even the new walk stops before 150 arcsec, because the emission along the major axis ends there
def reason(r):
    if r["why_now"] == [None, None]:
        return "no_gas"
    if r["nradii"] > FEW:
        return "dropped"
    if r["extent_bridge"] >= (FEW + 1) * RING_ARCSEC:
        return "gap_lt_beam"
    if r["r_out_new"] >= (FEW + 1) * RING_ARCSEC:
        return "gap_ge_beam"
    return "tail" if "tail" in r["why_new"] else "ends"

few = df[df["n_fit"] <= FEW].copy()
few["reason"] = few.apply(reason, axis=1)
REASONS = ["no_gas", "dropped", "gap_lt_beam", "gap_ge_beam", "tail", "ends"]
reasons = few.groupby("z")["reason"].value_counts().unstack(fill_value=0).reindex(columns=REASONS, fill_value=0)
reasons.loc["all"] = reasons.sum()
reasons

reason,no_gas,dropped,gap_lt_beam,gap_ge_beam,tail,ends
z,,,,,,
0.5,5,4,10,25,0,29
1.0,4,0,6,6,1,34
2.0,0,0,2,2,3,34
3.0,0,0,1,2,1,27
4.0,0,0,0,0,0,5
5.0,0,0,0,0,0,2
all,9,4,19,35,5,131


In [7]:
# the "dropped" galaxies: which of the asked rings BBarolo left out
# (it skips a ring when fewer than 20 of the ring's pixels are inside the mask: Bbarolo galfit.cpp:575-584)
#   dropped_inner: asked rings inside the first fitted ring (a central hole)
#   dropped_outer: asked rings beyond the last fitted ring
#   starts / stops: first and last emission along the major axis (left, right), arcsec, new rule
d = few[few["reason"] == "dropped"]
print(len(d), "galaxies:", (d["n_fit"] == 0).sum(), "with no ring fitted,", (d["dropped_inner"] > 0).sum(),
      "lost inner rings,", (d["dropped_outer"] > 0).sum(), "lost outer rings")
d[["model", "z", "subID", "nradii", "n_fit", "dropped_inner", "dropped_outer", "starts", "stops"]]

4 galaxies: 0 with no ring fitted, 4 lost inner rings, 0 lost outer rings


,model,z,subID,nradii,n_fit,dropped_inner,dropped_outer,starts,stops
910,CDM,0.5,61515,5,4.0,1.0,0.0,"[130, 50]","[690, 695]"
1232,CDM,0.5,53205,9,4.0,5.0,0.0,"[None, 150]","[None, 635]"
2718,SIDM1,0.5,59425,5,4.0,1.0,0.0,"[None, 25]","[None, 155]"
4075,vSIDM,0.5,71653,9,2.0,7.0,0.0,"[250, None]","[270, None]"


In [8]:
# with the new rule (task 6): rings from r_in + 15 arcsec out to r_out, one every 30 arcsec (n_new of them)
#   few_now:       4 fitted rings or fewer now
#   few_new:       would be asked for 4 rings or fewer with the new rule (BBarolo can still drop some)
#   edge_now:      the current walk reaches the cube border on a side (info.json extent_at_edge)
#   edge_new:      the new walk reaches the cube border on a side: these need a bigger cube
#   edge_new_only: edge_new but not edge_now: cubes to make bigger that the current rule never flagged
def after(g):
    edge_now = g["why_now"].apply(lambda w: "edge" in w)
    edge_new = g["why_new"].apply(lambda w: "edge" in w)
    return pd.Series(dict(N=len(g), few_now=(g["n_fit"] <= FEW).sum(), few_new=(g["n_new"] <= FEW).sum(),
                          edge_now=edge_now.sum(), edge_new=edge_new.sum(), edge_new_only=(edge_new & ~edge_now).sum()))

by_z_new = df.groupby("z").apply(after)
by_z_new.loc["all"] = after(df)
by_z_new

,N,few_now,few_new,edge_now,edge_new,edge_new_only
z,,,,,,
0.5,1424,73,37,6,89,83
1.0,1339,51,39,1,42,41
2.0,854,41,37,1,11,10
3.0,376,31,28,0,7,7
4.0,80,5,5,0,1,1
5.0,8,2,2,0,0,0
all,4081,203,148,8,150,142


In [9]:
# galaxies with 5 fitted rings or more now that would be asked for 4 or fewer with the new rule
# (the new rings start at r_in, the larger first-emission radius, instead of at the centre)
worse = df[(df["n_fit"] > FEW) & (df["n_new"] <= FEW)]
print(len(worse), "galaxies")
worse[["model", "z", "subID", "nradii", "n_fit", "dropped_inner", "starts", "stops", "r_in_new", "r_out_new", "n_new"]]

1 galaxies


,model,z,subID,nradii,n_fit,dropped_inner,starts,stops,r_in_new,r_out_new,n_new
868,CDM,0.5,26944,19,12.0,7.0,"[245, 460]","[595, 545]",460.0,595.0,4


In [10]:
# check of the new r_in rule: galaxies where one side's first emission lies beyond the end of the other side's gas,
# so with r_in = the larger first-emission radius the rings would start past all the gas on the other side
def starts_past_other_side(r):
    sides = [(a, b) for a, b in zip(r["starts"], r["stops"]) if a is not None]
    return len(sides) == 2 and max(a for a, b in sides) > min(b for a, b in sides)

odd = df[df.apply(starts_past_other_side, axis=1)]
print(len(odd), "galaxies")
odd[["model", "z", "subID", "starts", "stops", "r_in_new", "r_out_new", "n_new"]]

1 galaxies


,model,z,subID,starts,stops,r_in_new,r_out_new,n_new
1978,SIDM1,1.0,44087,"[75, 15]","[305, 70]",75.0,305.0,7


In [11]:
# one galaxy: '#' = pixel with emission, '.' = blank, from the centre outward (1 character = 5 arcsec)
# walks: (first emission, last emission) in pixels, and why the walk stopped
model, z, sub = "CDM", 0.5, 61515
for name, (m1, m0) in zip(("left", "right"), major_axis(model, z, sub)):
    print(name.ljust(6) + "".join("#" if np.isfinite(v) else "." for v in m1))
    print(" " * 6, "now:", walk_now(m1), " new:", walk_new(m1, m0))
print("fitted radii:", fitted_radii(model, z, sub))
df[(df["model"] == model) & (df["z"] == z) & (df["subID"] == sub)].T

left  ..........................######..#########################################################################################################
       now: (26, 31, 'gap<beam')  new: (26, 138, 'edge')
right ..........###########..#####################################################################################################################
       now: (10, 20, 'gap<beam')  new: (10, 139, 'edge')
fitted radii: [ 45.  75. 105. 135.]


,910
model,CDM
z,0.5
subID,61515
bbarolo_rc,0.0
nradii,5
n_fit,4.0
dropped_inner,1.0
dropped_outer,0.0
why_now,"[gap<beam, gap<beam]"
extent_now,155
